<a href="https://colab.research.google.com/github/datnguyen723/Train_NhanDienDongVat_CNN/blob/main/traincnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import shutil
import random


input_folder = "/content/drive/MyDrive/Dataset"
output_folder = "/content/drive/MyDrive/Split_Dataset"
val_count = 120

for split in ["train", "val"]:
    for category in os.listdir(input_folder):
        os.makedirs(os.path.join(output_folder, split, category), exist_ok=True)

for category in os.listdir(input_folder):
    images = os.listdir(os.path.join(input_folder, category))
    random.shuffle(images)

    val_images = images[:val_count]
    train_images = images[val_count:]

    for img in train_images:
        shutil.copy2(os.path.join(input_folder, category, img), os.path.join(output_folder, "train", category, img))
    for img in val_images:
        shutil.copy2(os.path.join(input_folder, category, img), os.path.join(output_folder, "val", category, img))

print("Dataset đã chia xong!")


Dataset đã chia xong!


In [ ]:

train_dir = '/content/drive/MyDrive/Split_Dataset/train'
val_dir = '/content/drive/MyDrive/Split_Dataset/val'
test_dir = '/content/drive/MyDrive/test'

In [ ]:
import os

def count_images(directory):
    class_counts = {}
    total = 0
    for category in os.listdir(directory):
        category_path = os.path.join(directory, category)
        if os.path.isdir(category_path):
            count = len(os.listdir(category_path))
            class_counts[category] = count
            total += count
    return class_counts, total


train_counts, train_total = count_images(train_dir)
val_counts, val_total = count_images(val_dir)
test_counts, test_total = count_images(test_dir)


print("Số lượng ảnh trong tập TRAIN:", train_counts, "Tổng:", train_total)
print("Số lượng ảnh trong tập VAL:", val_counts, "Tổng:", val_total)
print("Số lượng ảnh trong tập TEST:", test_counts, "Tổng:", test_total)


Số lượng ảnh trong tập TRAIN: {'cat': 560, 'chicken': 560, 'dog': 560, 'pig': 560} Tổng: 2240
Số lượng ảnh trong tập VAL: {'cat': 120, 'chicken': 120, 'dog': 120, 'pig': 120} Tổng: 480
Số lượng ảnh trong tập TEST: {'pig': 120, 'dog': 120, 'chicken': 120, 'cat': 120} Tổng: 480


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)


train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

test_generator = val_datagen.flow_from_directory(
    test_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 2233 images belonging to 4 classes.
Found 476 images belonging to 4 classes.
Found 478 images belonging to 4 classes.


In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       3,211,392 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │             516 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,305,156 (12.61 MB)

 Trainable params: 3,305,156 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.models import load_model

model = load_model('/content/drive/MyDrive/save_model/model_09-23_16-03-2025.keras')

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    train_generator,
    epochs=50,
    validation_data=val_generator,
    callbacks=[early_stop]
)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
 4/70 ━━━━━━━━━━━━━━━━━━━━ 8:47 8s/step - accuracy: 0.8776 - loss: 0.2243

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f'Test Loss: {test_loss:.4f}')

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


15/15 ━━━━━━━━━━━━━━━━━━━━ 144s 10s/step - accuracy: 0.8002 - loss: 0.4502
Test Accuracy: 84.31%
Test Loss: 0.3772


In [ ]:
from datetime import datetime
current_time = datetime.now().strftime("%H-%M_%d-%m-%Y")

model.save('/content/drive/MyDrive/save_model/model_'+current_time+'.keras')
print(f"Saved!")

Saved!


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')
plt.show()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')
plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image


test_class = random.choice(os.listdir(test_dir))
test_class_path = os.path.join(test_dir, test_class)

test_img_name = random.choice(os.listdir(test_class_path))
test_img_path = os.path.join(test_class_path, test_img_name)


img = image.load_img(test_img_path, target_size=(224, 224))
plt.imshow(img)
plt.axis("off")
plt.title(f"Ảnh thực tế: {test_class}")
plt.show()
